***Laue Patterns digital processing to get single pixel or roi-related counters value over a dataset of images***

***pixel intensity monitoring***

*** useful visualize grains in 2D Map (amesh dmesh scan) ***

*** no laue pattern indexing / refinement ***


Author: J.-S. Micha

Last Revision:   January   2026

tested with python3.12  ubuntu 24.04 jupyter-slurm,lbm32gpu1, visa.esrf.fr

**Objectives**

- Load and get pixel intensities on single pixel intensity or from ROIs statistics (pixel max intensity, position of max, postion of center of mass, position of gaussian fitting peak position)
- Visualize corresponding 2D or 1D scalar profile (intensity, spot position, etc)
- For 2D map (mesh scan): compilation of several map allows spatial determination of scattering region (grain, crystal, fluorescing material). 

Examples with following data:

- 3202812, Data at /data/bm32/inhouse/laue/data/32-02-812  (june 2018), Laue camera is sCMOS

- A 321216, Data at /data/visitor/a321216, Laue camera is sCMOS_4M

- blc15488  /data/projects/mapgrainxl/blc15488, Laue camera is sCMOS

# SET environment

In [ ]:
JUPYTER_LAB = True # jupyter lab (at ESRF)
JUPYTER_HUB = False # jupyter hub or notebook
JUPYTER_VSCODE = False #  jupyter on vscode

In [ ]:
# at ESRF during experiment and 90 days after at /data/visitor
# OR for a long term project at /data/projects
DATA_AT_ESRF_NICE=True   # False

# data can be /data/visitor or /data/projects  and  anywhere on linux ESRF machine or linux personnal laptop
ON_LINUX = True  # False

In [ ]:
import sys
print('Lauetools environment',sys.executable)

# IMPORT packages

In [ ]:
if JUPYTER_HUB : # jupyterhub
    %matplotlib notebook
elif JUPYTER_VSCODE: # vscode
    %matplotlib widget
elif JUPYTER_LAB: #  jupyterlab
    %matplotlib widget

import os, time, copy
import itertools
import glob
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd

import ipywidgets
from ipywidgets import FloatProgress, widgets, TwoByTwoLayout
from IPython.display import display
from ipywidgets import interact, interactive, fixed, interact_manual

from tqdm import tqdm, tqdm_notebook
import itertools
import multiprocessing
from multiprocessing import active_children, cpu_count
print(f'{multiprocessing.cpu_count()} cpu(s) are available!\n\n')
import fabio


In [ ]:
# Setting absolute path to LaueTools Modules if default version of the source code is not up to date
if 1:
    import sys
    #sys.path.insert(0,'/data/bm32/inhouse/STAFF/JSM/lauetools_devNotebooks/lauetools')
    #sys.path.insert(0,'/home/esrf/micha/lauetools_devNotebooks/lauetools')

    sys.path.insert(0,'/data/bm32/inhouse/lauetoolsenv2/lib/python3.12/site-packages')
    
    import LaueTools as LT
    print('code from', LT.__file__)

In [ ]:
import LaueTools.GUI.mosaic as MOS
import LaueTools.generaltools as GT
import LaueTools.IOimagefile as IOimage
import LaueTools.imageprocessing as Improc
import LaueTools.dict_LaueTools as DictLT
import LaueTools.IOLaueTools as IOLT
import LaueTools.readmccd as RMCCD
import LaueTools.blissdatafolderstructure as blissfolders
import LaueTools.logfile_reader as iohdf5

import LaueTools
print('you are using LaueTools of the following folder (of possibly an environment)\n', LaueTools.__file__)

import LaueTools.imagescollector as LTcollect
# wait for more recent version lauetools
collectpixelvalue_singlefile = LTcollect.collectpixelvalue_singlefile
collectroissum_singlefile = LTcollect.collectroissum_singlefile
collectroisptp_singlefile = LTcollect.collectroisptp_singlefile
collectroiarray_singlefile = LTcollect.collectroiarray_singlefile
collectroismax_singlefile = LTcollect.collectroismax_singlefile
collectroisXYmax_singlefile = LTcollect.collectroisXYmax_singlefile
collectroisXYcenterofmass_singlefile = LTcollect.collectroisXYcenterofmass_singlefile
collectroisfitpeak_singlefile = LTcollect.collectroisfitpeak_singlefile

try:
    collectrois_nbhotpixels = LTcollect.collectrois_nbhotpixels
except:
    fvbgs


In [ ]:
if 1: # test if we can get some plots (depending on notebook version ... and ipympl needs or not)
    fig_,ax_ = plt.subplots()
    ax_.plot(np.arange(20))

# SET user-defined `ExperimentFolder`

parent folder of all raw data ot the experiment. For data at ESRF during and after experiment, the path should contain 'RAW_DATA'

In [ ]:
#Experiment number or id?   the subfolder of date will be added if there is one
#expId = 'a321216'
expId = '322812'
expId = 'blc15488'
expId='a321219'
expId='blc16859'
expId='utr20'
expId='ma6758'  # Lhuissier et al Al DAXm March 2026

## Al Lhuissier 

In [ ]:
if expId == 'ma6758':  # test HERAUD
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder)
       
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

## TP hercyules utr20

In [ ]:
if expId == 'utr20':  # test HERAUD
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder)
       
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

## NiTi, Nacre

In [ ]:
if expId == 'blc16859':  # test HERAUD
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder)
       
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

## ZrO2 pillars

In [ ]:
if expId == 'a321219':
    print(expId)
    
    CCDLabel = 'EIGER_4MCdTe'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder)
       
    
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

## Al/Al2Cu 322812 data akamatsu et al

In [ ]:
if expId == '322812':
    
    CCDLabel = 'sCMOS'
    prefixfilename = 'ech28_'
    suffix = 'tif'
    
    userdefined_datafolder =  '/data/bm32/inhouse/data/laue/32-02-812'
    HDF5_LOGFILE_EXISTS = False

    ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
    else:
        GT.printred(f'\n"ExperimentFolder" does not exist ! : \n{ExperimentFolder}')

## Cr/Zr A321216 data Ribart et al

In [ ]:
if expId == 'a321216':
    print(expId)
    
    CCDLabel = 'sCMOS_4M'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder)
       
    
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

In [ ]:
if expId == 'blc15488':
    print(expId)
    
    CCDLabel = 'sCMOS'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    DATA_AT_ESRF_NICE = True
    HDF5_LOGFILE_EXISTS = True
    if DATA_AT_ESRF_NICE:
        
        #nicefolder = 'visitor'
        nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder, expDate='20240601')
        ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        #ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder)
       
    
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

In [ ]:
if expId == 'me1701':

    CCDLabel = 'sCMOS'  # default sCMOS  = quad  (4M)  or 'sCMOS_4M'  binned 3x3 36M IMAGESTAR
    
    userdefined_datafolder =  '/my/folder/to/data'
    HDF5_LOGFILE_EXISTS = True
    
    if DATA_AT_ESRF_NICE:
        
        nicefolder = 'visitor'
        #nicefolder = os.path.join('projects','mapgrainxl')
    
        #--------------------------------------------
        ExperimentFolder = os.path.join('/data',nicefolder, f'{expId}/bm32/')
    
        listdates = os.listdir(ExperimentFolder)
        print('possible dates',listdates)
        if len(listdates)>1:
            GT.printyellow(f'\nBe careful, there are several dates ... => {listdates}\n')
        
        # to uncomment two lines to precise the date if there are several ones
        # ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder, expDate='20250212')
        # ExperimentFolder = os.path.join(ExperimentFolder,'RAW_DATA')
        
        ExperimentFolder= blissfolders.setExperimentFolder_with_date(ExperimentFolder)
       
    
        print('ExperimentFolder set to ', ExperimentFolder)
    else:
        ExperimentFolder = userdefined_datafolder
        
    if os.path.exists(ExperimentFolder):
        GT.printgreen(f'\n"ExperimentFolder" exists ! : \n{ExperimentFolder}')
        
    if DATA_AT_ESRF_NICE and 'RAW_DATA' not in ExperimentFolder:
        GT.printyellow(f'\n"ExperimentFolder" does not contain "RAW_DATA"! Are you sure?')

# Thanks to HDF5 file, browsing data folders and building rapidly scan dictionnary `d`.

`d` to be used in next section

## list of folders and files

In [ ]:
if ON_LINUX:
    # uncomment the line to make the listing of a given folder
    # !ls /data/visitor/ma5820/bm32/20230304/SiCnuit/SiCnuit_bulle2/scan0001
    !ls {ExperimentFolder}

In [ ]:
if ON_LINUX:
    if DATA_AT_ESRF_NICE:
        print('expId',expId)
        !tree -Lt 2 -d {ExperimentFolder}
    else:
        !ls {os.path.join(ExperimentFolder,'RAW_DATA')}

In [ ]:
if ON_LINUX:
    print('expId',expId)
    !tree -Lt 3 -d {ExperimentFolder}

In [ ]:
if ON_LINUX:
    # uncomment the line to make the listing of a given folder
    # !ls /data/visitor/ma5820/bm32/20230304/SiCnuit/SiCnuit_bulle2/scan0001
    #!ls {os.path.join(ExperimentFolder,'echB/echB_mapfluo/scan0043')}
    !ls {os.path.join(ExperimentFolder,'')}

In [ ]:
# catch main hdf5 file
if HDF5_LOGFILE_EXISTS:

    defaultindex_h5file = 0

    pathHDF5 = None
    if not DATA_AT_ESRF_NICE:
        hdf5folder = ExperimentFolder
    hdf5filelist = []
    for elem in os.listdir(ExperimentFolder):
        
        if elem.endswith('h5'):
            pathHDF5 = os.path.join(ExperimentFolder, elem)
            hdf5filelist.append(pathHDF5)
            
    pathHDF5 = hdf5filelist[defaultindex_h5file]

    if pathHDF5 and len(hdf5filelist)>=1:
        GT.printgreen(f'!! Found main HDF5 file: {hdf5filelist}')
        if len(hdf5filelist)>1:
            GT.printyellow(f'Choose one .h5 to set the variable "PathHDF5" by selecting the proper index "defaultindex_h5file" in the list above')
        print(f'\nNow "PathHDF5" is set to {pathHDF5}')
    else:
        GT.printred('HDF5 file not found !!!')

## READ or UPDATE the hdf5 file: 

- set `listscans`: list of scans 1D or 2D
- set  `listcts` : list of ct
- build a pandas object for all scans `dfallscans`

In [ ]:
# it can take a while
if HDF5_LOGFILE_EXISTS:
    print('pathHDF5',pathHDF5)
    listscans, listcts = iohdf5.get_scans_cts(pathHDF5)
    print("number scans found :",len(listscans))
    print("number saved count commands found: ", len(listcts))
    # if len(listcts)>0:
    
    #     print("example of first count item : \n", listcts[0])


### Location of all scans

buiid a pandas dataframe `dfallscans`

In [ ]:
# it can take a while
dfallscans = None
if HDF5_LOGFILE_EXISTS:
    import pandas as pd


    pd.set_option('display.max_rows', None)
    dfallscans = pd.DataFrame(listscans, columns=['start_time', 'end_time','sample_dataset_scanindex', 'fullcommand',
                                              'scanindex','scantype','motors','localhdf5file','imagefolder'])
dfallscans

In [ ]:
if 1:# example to see in detail a specific item
    item_idx = 48
    #item_idx = len(dfallscans)-1  #last item!
    print(f'------ dfallscans item index {item_idx} -----------')
    print(dfallscans.iloc[item_idx])
    # to get value of a given column and item
    dfallscans['imagefolder'][item_idx]

## BUILD a dictionnary of scan parameters

- select the item/scan index
- create a dict  

In [ ]:
# construct automatically dict or scan parameters from item of dfallscans given index
if expId == 'ma6758':
    d25 = iohdf5.build_dict_scan(25, dfallscans, CCDLabel=CCDLabel)
    d26 = iohdf5.build_dict_scan(26, dfallscans, CCDLabel=CCDLabel)
    d48 = iohdf5.build_dict_scan(48, dfallscans, CCDLabel=CCDLabel)

    
if expId == 'utr20':
    d2 = iohdf5.build_dict_scan(2, dfallscans, CCDLabel=CCDLabel)
    d5 = iohdf5.build_dict_scan(5, dfallscans, CCDLabel=CCDLabel)
if expId == 'blc16859':
    # NiTi
    d11 = iohdf5.build_dict_scan(11, dfallscans, CCDLabel=CCDLabel)
    d10 = iohdf5.build_dict_scan(10, dfallscans, CCDLabel=CCDLabel)
    d13 = iohdf5.build_dict_scan(13, dfallscans, CCDLabel=CCDLabel)
    d16 = iohdf5.build_dict_scan(16, dfallscans, CCDLabel=CCDLabel)
    d76 = iohdf5.build_dict_scan(76, dfallscans, CCDLabel=CCDLabel)
    # nqcre
    d90 = iohdf5.build_dict_scan(90, dfallscans, CCDLabel=CCDLabel)
    d102 = iohdf5.build_dict_scan(102, dfallscans, CCDLabel=CCDLabel)
    # Texier
    d105 = iohdf5.build_dict_scan(105, dfallscans, CCDLabel=CCDLabel)
    d107 = iohdf5.build_dict_scan(107, dfallscans, CCDLabel=CCDLabel)

if expId == 'a321219':
    d42 = iohdf5.build_dict_scan(42, dfallscans, CCDLabel=CCDLabel) 
    d41 = iohdf5.build_dict_scan(41, dfallscans, CCDLabel=CCDLabel)   

if expId == 'hc6287':
    d20 = iohdf5.build_dict_scan(20, dfallscans, CCDLabel=CCDLabel) 
    d25 = iohdf5.build_dict_scan(25, dfallscans, CCDLabel=CCDLabel)     
    d29 = iohdf5.build_dict_scan(29, dfallscans, CCDLabel=CCDLabel)

    d33 = iohdf5.build_dict_scan(33, dfallscans, CCDLabel=CCDLabel)
    d34 = iohdf5.build_dict_scan(34, dfallscans, CCDLabel=CCDLabel)
    d35 = iohdf5.build_dict_scan(35, dfallscans, CCDLabel=CCDLabel)
    # sunday
    d39 = iohdf5.build_dict_scan(39, dfallscans, CCDLabel=CCDLabel)
    d41 = iohdf5.build_dict_scan(41, dfallscans, CCDLabel=CCDLabel)
    
    dd = d20
    
    print(f"full command for scan {dd['sample_dataset_scanindex']} is {dd['fullcommand']}")

if expId == 'a321216':
    d0 = iohdf5.build_dict_scan(0, dfallscans, CCDLabel=CCDLabel)
    d1 = iohdf5.build_dict_scan(1, dfallscans, CCDLabel=CCDLabel)
    d2 = iohdf5.build_dict_scan(2, dfallscans, CCDLabel=CCDLabel)
    print(d0['fullcommand'])
    print(d1['fullcommand'])

    dd = d0
    print(f"full command for scan {dd['sample_dataset_scanindex']} is {dd['fullcommand']}")

if expId == 'blc15488':
    d10 = iohdf5.build_dict_scan(10, dfallscans, CCDLabel=CCDLabel)

    dd=d10
    print(f"full command for scan {dd['sample_dataset_scanindex']} is:\n{dd['fullcommand']}")

if expId == 'me1701':

    
    d315 = iohdf5.build_dict_scan(315, dfallscans, CCDLabel='sCMOS')  # 3mol% Yttria stabilized Zirconia and Alumina layer: /scan0026/img_0210.tif 
    d289 = iohdf5.build_dict_scan(289, dfallscans, CCDLabel='sCMOS')  # Ge: /data/visitor/me1701/bm32/20241113/RAW_DATA/A45ZTAA/A45ZTAA_Gedaxm GeDaxm
    d290 = iohdf5.build_dict_scan(290, dfallscans, CCDLabel='sCMOS') # Alumina layer:  /scan0001/img_0005.tif   
    
    d490 = iohdf5.build_dict_scan(490, dfallscans, CCDLabel='sCMOS')  # 3mol% Yttria stabilized Zirconia and Alumina layer: /scan0001/img_0001.tif 
    d515 = iohdf5.build_dict_scan(515, dfallscans, CCDLabel='sCMOS')  # Alumina layer: /scan0026/img_0250.tif     
    d489 = iohdf5.build_dict_scan(489, dfallscans, CCDLabel='sCMOS')  # Ge: /data/visitor/me1701/bm32/20241113/RAW_DATA/Cshape45/Cshape45_Gedaxm
    
    d40 = iohdf5.build_dict_scan(40, dfallscans, CCDLabel='sCMOS')  # map
    
    dd= d290
    print(f"full command for scan {dd['sample_dataset_scanindex']} is:\n{dd['fullcommand']}")
    print(dd)

In [ ]:
d48

## [OPTION] HDF5 mining to look for specific scans

### Location of mesh scans (2D map)

In [ ]:
dfmeshscans = None
if HDF5_LOGFILE_EXISTS:
    # ALL MESH SCANS  (2D Map)
    pd.set_option('max_colwidth', 200)
    dfmeshscans = dfallscans[dfallscans['scantype']=='amesh'][['sample_dataset_scanindex','fullcommand', 'imagefolder', 'motors']]
dfmeshscans

In [ ]:
# to see a specific item  use ** iloc **  (absolute index in dfmeshscans)
item_idx = 2

if item_idx<len(dfmeshscans):
    print(dfmeshscans.iloc[item_idx])
else:
    GT.printyellow('this item index does not exist')

### Location of zf or thf scans (DAXM)

In [ ]:
# ALL ZF SCANS (DAXM Wire scans) or Diamond scan (thf)
if HDF5_LOGFILE_EXISTS:

    motorname = 'zf'
    print('location of images for scans of %s'%motorname)
    dfallscans.loc[dfallscans['motors'] == motorname]['imagefolder']
    
dfallscans

In [ ]:
dfzfscans = None
if HDF5_LOGFILE_EXISTS:

    pd.set_option('max_colwidth', 200)
    dfzfscans = dfallscans[dfallscans['motors']=='zf'][['sample_dataset_scanindex','fullcommand', 'imagefolder', 'motors']]
dfzfscans

### Location of scans for specified sample

In [ ]:
# narrow the search of specific folder

selected_scans = None

#df = dfzfscans
df = dfallscans

if expId == 'me1701':
    str_in_dataset = 'A45'  #'TRUE_COM' #'A45'
    motor_in_motors =  'zf' #'thf'
if expId == 'a321216':
    str_in_dataset = 'TRUE_COM' 
    motor_in_motors =  'thf'
if expId == 'blc15488':
    str_in_dataset = 'ech15_map2Dexpo' 
    motor_in_motors =  'xech'

cond_dataset  = df['sample_dataset_scanindex'].str.contains(str_in_dataset)
cond_motor  = df['fullcommand'].str.contains(motor_in_motors)

cond = cond_dataset & cond_motor

if HDF5_LOGFILE_EXISTS:
    selected_scans=df[cond][
                        ['sample_dataset_scanindex','fullcommand', 'imagefolder', 'motors']]

selected_scans

### retrieve bliss command and data from image and `dfallscans` from hdf5 file

In [ ]:
if expId == 'me1701':
    queryimage = os.path.join(d290['folder'],'img_0400.tif')
    print('image', queryimage)

    blisscommand, imagedate, dfallscans_index = IOimage.fromscmosdate2blisscommand(queryimage, dfallscans)
    print('blisscommand',blisscommand)
    print('imagedate',imagedate)
    print('dfallscans_index', dfallscans_index)

#### Example 1: Select the scan collecting images at querydate

In [ ]:
import datetime

querydate = datetime.datetime(2024,6,18,23,33,30)  # y m d  h min sec
print('querydate',querydate)

mask = (dfallscans['end_time'] > querydate.isoformat()) & (dfallscans['start_time'] <= querydate.isoformat())
dfallscansgooddate = dfallscans.loc[mask]

print(dfallscansgooddate.fullcommand.values)
dfallscansgooddate

#### Example 2: Select the scan collecting images at during the period

In [ ]:
# period is defined by  [querydate-timespan, querydate+timespan]
querydate = datetime.datetime(2024,6,18,23,33,30)  # y m d  h min sec
timespan = 60 # min

minidate = datetime.datetime.fromtimestamp(querydate.timestamp()-timespan*60)
maxidate = datetime.datetime.fromtimestamp(querydate.timestamp()+timespan*60)
mask2 = (dfallscans['end_time'] < maxidate.isoformat()) & (dfallscans['start_time'] > minidate.isoformat() )
dfallscansgoodperiod = dfallscans.loc[mask2]

#print(dfallscansgoodperiod.fullcommand.values)
print(dfallscansgoodperiod[['fullcommand', 'imagefolder', 'start_time']])
dfallscansgoodperiod

# SET `d` dict of experimental parameters for 1D or 2D map (scan, mesh)

In [ ]:
#Manual entry
if expId == '322812':
    #ExperimentFolder#
    # Al/Al2Cu
    #"/data/bm32/inhouse/data/laue/32-02-812"
    # MAP 2D
    # amesh xech ... ... yech .......
    mapdimensions=(17,17)   # = (nb of steps+1, nb of steps +1 )
    dictmap2D_322812={
    'folder':os.path.join(ExperimentFolder,''),
        'prefix': 'ech28_',
        'listindices' : np.arange(0,mapdimensions[0]*mapdimensions[1]),
        'mapdimensions': mapdimensions,
        'CCDLabel' :'sCMOS' ,
        'peaklistfile': None,
        'collector': 'pixelval',
        'nbimagesperline': mapdimensions[0],
        'fastaxis':'xech',  # motor that moves first to form the first line
        'slowaxis':'yech', # motor that moves once to go from one line to the other
        'scantype':'map',
    }

    dictmap2D_322812['suffix']=suffix

if expId == '322812':
    d =dictmap2D_322812
    
# ********** from hdf5 file **************
if expId == 'a321216':
    d =d0
if expId == 'blc15488':
    d =d10
if expId == 'hc6287':
    d = dd


if expId == 'me1701':
    d= d289  # Ge
    #
    d= d290   # alumina

if expId == 'a321219':
    d= d41 

if expId == 'blc16859':
    d= d11
    d=d13
    d=d16
    # NiYi
    d=d76
    #nacre
    d=d90
    d=d102
    d=d105
    d=d107

if expId == 'utr20':
    d= d5

if expId == 'ma6758':
    d= d25
    d=d26

print('selected dict is:')
print(d)

# DEFINE ROI locations (2D pixel coordinates on detector)

- peaks from peaksearch (on a given image)
- evenly spaced grid of points

## Peaksearch on single image

float coordinates are converted to integer coordinates

### compute `roiposition` from peaksearch

In [ ]:
#Manual entry
if expId == '322812':
    imageindex=0  # Al phase
    #imageindex=6  # Al2Cu phase
    IntensityThreshold = 500
if expId == 'me1701':
    imageindex=0
    IntensityThreshold = 2000 #for Al2O3 phase
if expId == 'hc6287':
    imageindex=0
else:
    imageindex=0
    IntensityThreshold = 3000  # 200 for Ge wafer

#--------------end of user input--------------------
if isinstance(d['folder'],list):
    folder = d['folder'][0]
else:
    folder = d['folder']
intensities = None
roiposition = None
r = RMCCD.PeakSearch(os.path.join(folder,f'{d['prefix']}%04d.{d['suffix']}'%imageindex),
                    CCDLabel=d['CCDLabel'],
                    IntensityThreshold=IntensityThreshold,
                    Data_for_localMaxima='auto_background',
                    fit_peaks_gaussian=0,
                    local_maxima_search_method=0,
                    boxsize=15,
                    Saturation_value= 600000,  
                    PeakSizeRange=(.3, 20),
                    maxPixelDistanceRejection=10,
                    NumberMaxofFits=10000)  # high value to for 1000s spots
if r is None:
    GT.printyellow('Nothing found. Try to change "IntensityThreshold" neither too small nor too large (first: 1000-5000 ?)')
if r is not None:
    roiposition = r[-1]
    nb_peaks_found = len(roiposition)
    if nb_peaks_found>0:
        GT.printgreen(f'nb peaks {len(roiposition)}')
    intensities = r[0][:,2]
    
    #hottest pixel:
    hottestspot_index = np.argsort(r[0][:,2])[::-1][0]
    print('hottest pixel : [X, Y, intensity above background] is:\n', r[0][hottestspot_index][:3])

In [ ]:
# OPTION: mask some regions
MASK_REGIONS = False
if MASK_REGIONS:
    XY= roiposition
    #center of rectangle
    #Xtest, Ytest=[2324,3521]
    Xtest, Ytest=[2226,3670]
    # --- filtering -----
    xcond =np.fabs(XY[:,0]-Xtest)<5
    ycond =np.fabs(XY[:,1]-Ytest)<10
    cond = np.logical_and(xcond,ycond)
    np.where(cond)

In [ ]:
SORTED_BY_INTENSITY = True
if SORTED_BY_INTENSITY:# sorted by decreasing intensity
    sortedindices = np.argsort(r[0][:,2])[::-1]
    speaks = r[0][sortedindices]
    roiposition = speaks[:,:2]
    intensities = r[0][:,2][sortedindices]
    print('   X, \t\t\tY, \t\tIntensity above BackGround')
    print(np.column_stack((roiposition, intensities)))

### [OPTION] GUI: laue pattern plot browser

- displays image
- shows `roiposition` from peaksearch results

In [ ]:
imageindex = 3

fullpath = os.path.join(folder,f'{prefixfilename}%04d.{d['suffix']}'%imageindex)
if not os.path.exists(fullpath):
    GT.printyellow(f'\n"fullpath" does not exist ! : \n{fullpath}')
    something_to_stop

with fabio.open(fullpath) as img:
    imgdata = img.data
    
print('imgdata.shape', imgdata.shape, '\n\n')
    
fig,ax = plt.subplots()
#fig.suptitle('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))
# #ax.imshow(np.log10(imgdata), vmin = 3, vmax = 3.5, cmap=plt.cm.inferno)
ax.imshow(imgdata, vmin = 1000, vmax =6000, cmap=plt.cm.OrRd)
# for pt in roiposition:
#     ax.scatter(pt[0],pt[1],marker='+',color='r')
    
def plotimage(imageindex=imageindex,vmin=1000, vmax=2000, show_rois=False):
    # TO IMPROVE  see wrokflow JSM   laueimproc
    #print('showroi', show_rois)
    ymin, ymax = ax.get_ylim()
    xmin, xmax = ax.get_xlim()

    ax.clear()
    
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    
    
    ymin, ymax, xmin, xmax = int(ymin),int(ymax),int(xmin),int(xmax)
    if ymin>ymax:
        yminc = ymin
        ymin = ymax
        ymax = yminc
    fullpath = os.path.join(folder,f'{prefixfilename}%04d.{d['suffix']}'%imageindex)
    with fabio.open(fullpath) as img:
        imgdata = img.data
        #print('imgdata.shape', imgdata.shape, '\n\n')
    if 'RAW_DATA' in fullpath:
        figtitle=fullpath.rsplit('/RAW_DATA/')[1]
    else:
        figtitle = fullpath
    fig.suptitle('%s\n%s'%(ExperimentFolder,figtitle))
    ax.imshow(imgdata, vmin = vmin, vmax =vmax, cmap=plt.cm.BuGn)
    if show_rois:
        #print('show rois!')
        for pt in roiposition:
            ax.scatter(pt[0],pt[1],marker='+',color='r')
            #ax.set_title('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))
    plt.show()

indexmax = max(d['listindices'])
indexmax =  d['mapdimensions'][0] *  d['mapdimensions'][1]-1
interactive(plotimage, imageindex=(0,indexmax),
            vmin=(500,1010),vmax=(1015,100000), show_rois=[False, True])

In [ ]:
# [OPTION] [HELPER] find the position of a roi given XY pixels coordinates
if 0:
    XY= roiposition
    #Xtest, Ytest=[2324,3521]
    Xtest, Ytest=[1156,1825]
    Xtest, Ytest=[918,1730]
    
    xtolerance, ytolerance = 5, 10
    
    #--------------------------------------
    xcond =np.fabs(XY[:,0]-Xtest)<xtolerance
    ycond =np.fabs(XY[:,1]-Ytest)<ytolerance
    cond = np.logical_and(xcond,ycond)
    potential_index = np.where(cond)[0]
    
    potential_index, XY[potential_index]

#### useful tools to sort spots (according to Y) for DAXM (and diamond scan)

In [ ]:
SORT_BY_Y=False

# FOR Diamond could be useful (scan thf)
# SORT_BY_Y = True

if d['scantype']=='daxm':  # mandatory tof the next algorithms
    GT.printgreen(f'DAXM scan!')
    SORT_BY_Y = True

#--------------------------------
if SORT_BY_Y:
    # sort by increasing Y
    sortedYindices = np.argsort(roiposition[:,1])
    roiposition= np.take(roiposition,sortedYindices, axis=0)
    intensities = np.take(intensities,sortedYindices, axis=0)
    GT.printgreen(f'Sorted {len(roiposition)} peaks by intensity increasing pixel Y in "roiposition" ')

#### Modify ROIs list

In [ ]:
MANUAL_MODIFY= False
if MANUAL_MODIFY:# manual modification of roi (overwrite)
    roiposition[-1]=np.array([744,700])
#     roiposition[1]=np.array([1116,853])
#     roiposition[2]=np.array([424,1936])

In [ ]:
MANUAL_ADD= False
if MANUAL_ADD:
    roiposition= np.concatenate((np.array([[744,700],[900,958],[550,360]]),roiposition), axis=0)
    # you may need to add elements `intensities` array
    intensities= np.concatenate((np.array([1,1,1]),intensities), axis=0)

In [ ]:
print('total number of spots in list: ',len(roiposition))
np.set_printoptions(suppress=True,precision=3)
print('   X,      Y,      Intensity above BackGround')
print(np.column_stack((roiposition, intensities)))
np.set_printoptions(suppress=True,precision=7)

## Get roi positions from a regular 2D Grid of points

set `gridroi`  ROI position

In [ ]:
# ROIs are in whole detector area 
#gridroi = GT.buildgridroi(spacing=50,maxnbpixels=2000)

# ROIs are in the half lower part of detector area 
gridroi = GT.buildgridroi_bottom(spacing=75,maxnbpixels=2050)
#gridroi = GT.buildgridroi_bottom(spacing=150,maxnbpixels=2000)
# set general parameter (which is filled  by peaksearch)
intensities_gridrois = None

gridroi, gridroi.shape

In [ ]:
# plot if needeed
nbrois_grid = gridroi.shape[0]
if nbrois_grid<1000:
    fig,ax = plt.subplots()

    try:
        if imgdata is None:
            imgdata = 1000*np.ones((2000,2000))
    except NameError:
        imgdata = 1000*np.ones((2000,2000))
    #ax.imshow(np.log10(imgdata), vmin = 3, vmax = 3.5, cmap=plt.cm.inferno)
    ax.imshow(imgdata, vmin = 1000, vmax =6000, cmap=plt.cm.OrRd)
    for pt in gridroi:
        ax.scatter(pt[0],pt[1],marker='+',color='r')
        ax.set_title('')
    plt.show()
else:
    GT.printyellow(f'nb of rois in the grid ({nbrois_grid}) is too large to be reasonnably plot')

# SET centers of ROI

- setting `d['peaklist']`

In [ ]:
ROISelection = {"CHOICE": "USE_GRIDROI_LIST"}

dropdown = widgets.Dropdown(
    options=["USE_PEAKSEARCH_LIST", "USE_GRIDROI_LIST"],value=ROISelection["CHOICE"],
    description="choice:")

def update(change):
    ROISelection["CHOICE"] = change["new"]
    print("Current choice =", ROISelection["CHOICE"])
dropdown.observe(update, names="value")

display(dropdown)

In [ ]:
choice = ROISelection["CHOICE"]

importlist=False
if choice == 'USE_GRIDROI_LIST':
    print('you choosed gridroi')
    # regular spaced ROIS
    try:
        iohdf5.getpeaklist(d, data=gridroi)
        importlist=True
    except NameError:
        print('gridroi not defined, please build a grid first\n')
    d['intensities']=None
    print(d['intensities'])
elif choice == 'USE_PEAKSEARCH_LIST':
    print('you choosed USE_PEAKSEARCH_LIST')
    # ROIS from peak peaksearch above
    try:
        iohdf5.getpeaklist(d, data=roiposition)
        d['intensities']=intensities
        importlist=True
    except NameError:
        print('roiposition, not defined, please do a peaksearch first\n')
    try:
        if intensities is not None:
            d['intensities']=intensities
            print('values of last peaksearch peaks intensities are in d["intensities"]')
    except NameError:
        pass
    

if importlist: 
    GT.printgreen(f"\nnb of rois : {len(d['peaklist'])}")
    print('current peaklist 5 first elements ', d['peaklist'][:5])
    


In [ ]:
## final dict for exp. parameters (folder and rois)
#print('final dict is:', d)
d['intensities']

# COLLECT Mosaic Multiprocessing 

each 2D small part around the same single pixel position `d['roicenter']`is taken over all images and rearranged to form form a mosaic of local 2D signal

## SET center of ROI with 2D pixel coordinates

Set `d['roicenter']`

In [ ]:
# SET center of the ROI with 2D pixelcoordinates

if expId == '322812':
    Al2Cuspot = [870,1830]
    Alspot = [1280,740]

    d['roicenter']= d['peaklist'][0]
    d['roicenter']= Al2Cuspot
if expId == 'blc15488':
    d['roicenter']= d['peaklist'][20]

## SET boxsize and [set of images]

- boxsize_X, boxsize_Y are half box size in X and resp. Y pixel detector directions
  
  Preferably, they must be odd (2p+1), so that full size of the ROI is 2*boxsize_X+1, 2*boxsize_Y+1 and the roi is centered on the roi input center.

In [ ]:
if __name__=='__main__':
    
    collector = 'mosaic'
    nbcpus= 120 # > 1 !!
    
    # for roimax or roiXYmax
    boxsize_X=13  # along pixel X   half size
    boxsize_Y=19  # along pixel Y   half size
    
    # images to be handled
    listindices = None#  None means take indices in d[['listindices']]
    nbcompletelines = None #10  # None all lines when scan is completed

    #---------------------end of user inputs-----------------------------------------------
    #---------------------Multiprocessing-----------------------------------------------
    dictparam=d

    maxnbcpus = cpu_count()
    if nbcpus is None:
        nbcpus = maxnbcpus
    else:
        nbcpus = max(2,min(nbcpus,maxnbcpus))
    print('nbcpus', nbcpus)
    
    roicenter=dictparam.get('roicenter', None)
    prefix=dictparam.get('prefix', None)
    folder=dictparam.get('folder', None)
    CCDLabel=dictparam.get('CCDLabel', None)

    if roicenter is None:
        GT.printred(f"d['roicenter'] is not set to extract pixel intensities around it")
    else:
        xroi, yroi = d['roicenter']
        maxdimension = 2015
        if xroi-boxsize_X<0 or xroi+boxsize_X>maxdimension or yroi-boxsize_Y<0 or yroi+boxsize_Y>maxdimension:
            GT.printred(f"d['roicenter'] {d['roicenter']} is too close from detector frame border for the boxsize : {(boxsize_X, boxsize_Y)}")
    
    if listindices is None:
        listindices=dictparam['listindices']
        if nbcompletelines is not None:
            listindices=dictparam['listindices'][:dictparam['nbimagesperline']*nbcompletelines]
            print(dictparam['mapdimensions'][0],nbcompletelines)
            d['mapdimensions']=(dictparam['mapdimensions'][0],nbcompletelines)
            d['listindices'] =listindices
            
    if not isinstance(folder, list):  # iteration over images on the same folder
        nbimages = len(listindices)
        listfolders = itertools.repeat(folder)
        
    else: # iteration over a list of folders
        print('Multiple folders screened')
        nbimages = len(folder)
        listfolders = folder
        listindices = itertools.repeat(dictparam['listindices'])
    
    #----------------------------------
    
    t00 = time.time()
    print(f'using {collector} as collector: {nbimages} images, {nbcpus} cpu(s)')
    
    args_mosaic = zip(listindices,
                   itertools.repeat(roicenter),
                   itertools.repeat(prefix),
                   listfolders,
                  itertools.repeat(boxsize_X),
                  itertools.repeat(boxsize_Y),
                   itertools.repeat(CCDLabel),
                   )
    
    #print('args_pixelvalue',[elem for elem in args_pixelvalue])
    
    with multiprocessing.Pool(nbcpus) as pool:
            
        allresults = pool.starmap(collectroiarray_singlefile,
                                      tqdm(args_mosaic, total=nbimages,
                                      desc='sum values progress bar:'), chunksize=1)

    allresults = np.array(allresults)

    elapsedtime=time.time()-t00
    print(f'total time is {elapsedtime:.3f} sec for {nbimages} images and {nbcpus} cpu(s)')

    children = active_children()
    print(f'Active children: {len(children)}')

    d['allresults']=allresults
    GT.printgreen('Results are in allresults or allresults key...\n\n!!! ----- Collection done ;-) ----!!')


GT.printgreen('Arranging collected data...\n\n!!!')
print('allresults.shape = (nbimages x boxsize x boxsize)', allresults.shape)
# plot mosaic from 2D map
if d['scantype']=='map':
    # processing and rearranging collected ROIs imagelets
    dimfast, dimslow = d['mapdimensions']
    print('axis dimensions: dimslow, dimfast',dimslow, dimfast)
    mosaic = np.zeros((dimslow, dimfast, 2*boxsize_Y+1, 2*boxsize_X+1))
    print('mosaic.shape', mosaic.shape)
    dict_map_imageindex ={}

    #print(d['listindices'])
    
    sm = mosaic.shape
    bigimage = np.zeros((sm[0]*sm[2],sm[1]*sm[3]))
    
    
    if dimfast > 0:
        for map_imageindex, absolute_imageindex in enumerate(d['listindices']):
            
            imap, jmap = map_imageindex // dimfast, map_imageindex % dimfast
    
            dict_map_imageindex[map_imageindex] = [absolute_imageindex,
                                                    map_imageindex,
                                                    imap, jmap]
            
            raw = allresults[map_imageindex,:,:]
            
            #datcrop = np.flipud(raw).T
            datcrop = raw
            
            mosaic[imap,jmap] = datcrop #datcrop.T #np.flipud(datcrop).T
            # for 2D map
            bigimage[imap*sm[2]:(imap+1)*sm[2], jmap*sm[3]:(jmap+1)*sm[3]] = np.flipud(datcrop)
    
    if 1:
        mosaictranspose = mosaic.transpose((0, 3, 1, 2))
        mosaicflat = mosaictranspose.reshape((dimfast * (2 * boxsize_X + 1), dimslow * (2 * boxsize_Y + 1)))
    
    print('mosaictranspose.shape',mosaictranspose.shape)
    print('mosaicflat.shape',mosaicflat.shape)
    #dat_dims = (nb_lines * (2 * boxsize_Y + 1), nb_col * (2 * boxsize_col + 1))
    GT.printgreen('Arranging Done !!!')

In [ ]:
print(bigimage.shape) # "Ypixel * slow dim", Xpixel * fast dim"
print(mosaic.shape) # slow dim, fast dim, Ypixel, Xpixel

## GUI Imagelet browser as a function of imageindex

In [ ]:
# interactive plot
imageindex0 = 3
dimfast = d['mapdimensions'][0]
roicenter = d['roicenter']
i_x0, i_y0 = imageindex0//dimfast,imageindex0%dimfast  # slow, fast

#print(i_x0, i_y0)

figt, axt = plt.subplots()
axt.imshow(mosaic[i_x0, i_y0], origin='upper')

slider_ix = widgets.IntSlider(value=i_x0, min=0, max=mosaic.shape[0]-1, step=1)
slider_iy = widgets.IntSlider(value=i_y0, min=0, max=mosaic.shape[1]-1, step=1)
slider_vmin = widgets.IntSlider(value=1000, min=-100, max=20000, step=1)
slider_vmax = widgets.IntSlider(value=3000, min=1101, max=60000, step=1)
output = widgets.Output()

def handle_slider_change(change):
    with output:
        output.clear_output()
        print(f"The new slider value is: {change.new}")

slider_ix.observe(handle_slider_change, 'value')
slider_iy.observe(handle_slider_change, 'value')

widgets.VBox([slider_ix,slider_iy, slider_vmin, slider_vmax, output])

def plotroi(i_slow, i_fast, vmin=1000, vmax=4000, scale='linear'):
    ymin, ymax = axt.get_ylim()
    xmin, xmax = axt.get_xlim()

    i_x, i_y = i_slow, i_fast

    axt.clear()
    
    axt.set_xlim(xmin, xmax)
    axt.set_ylim(ymin, ymax)
    
    
    ymin, ymax, xmin, xmax = int(ymin),int(ymax),int(xmin),int(xmax)
    if ymin>ymax:
        yminc = ymin
        ymin = ymax
        ymax = yminc

    imageindex = i_x*dimfast+i_y

    if scale == 'linear':
        dplot = mosaic[i_x, i_y]
        vminplot = vmin
        vmaxplot = vmax
    elif scale == 'log':
        dplot = np.log10(mosaic[i_x, i_y])
        vminplot = np.log10(vmin)
        vmaxplot = np.log10(vmax)
            
    axt.imshow(dplot, origin='upper', vmin=vminplot, vmax=vmaxplot)
    axt.set_xlabel('pixel X')
    axt.set_ylabel('pixel Y')
    tt = f'roicenter at pixel ({roicenter[0]}, {roicenter[1]}) '
    tt += f'image index {imageindex}'
    axt.set_title(tt)

    print('max:', np.amax(dplot))
    print('min:', np.amin(dplot))

www = widgets.interact(plotroi, i_slow=slider_ix,i_fast=slider_iy,
                 vmin = slider_vmin, vmax = slider_vmax, scale=['linear', 'log'])
#print(f'i_slow // {d["slowaxis"]}   i_fast // {d["fastaxis"]}')

## PLOT mosaic of an imagelet from a 2D scan (map)

In [ ]:
# plot mosaic from 2D map
vmin = 1000
vmax = 4000
if expId == 'blc15488':
    vmin = 1600
    vmax = 2000
folder = d['folder']  #d['imagefolder']

if d['scantype']=='map':
    #bigimage.shape # "Ypixel * slow", Xpixel * fast"
    #mosaic.shape # slow , fast , Ypixel, Xpixel
    #print('mosaic.shape', mosaic.shape)
    def format_coord(x, y):
        col = int(x)
        row = int(y)
        if col >= 0 and col < bigimage.shape[1] and row >= 0 and row < bigimage.shape[0]:
            #cnt_idx = fig.gca()
            i, j = col//mosaic.shape[3], row//mosaic.shape[2]
            img_idx = 0+ mosaic.shape[1]*j+i
            return "x=%1.4f, y=%1.4f, imageid=%d" % (x, y,img_idx)
        else:
            return "x=%1.4f, y=%1.4f" % (x, y)
    
    print('bigimage.shape', bigimage.shape)
    figmosaic, axmosaic = plt.subplots(figsize=(6,4))
    axmosaic.imshow(bigimage, origin='lower', vmin=vmin,vmax=vmax)
    if d['fastaxis']=='yech':
        axmosaic.set_xlabel(f"fastaxis {d['fastaxis']} // pixelY")
        axmosaic.set_ylabel(f"slowaxis {d['slowaxis']} // pixelX")
    else:
        axmosaic.set_xlabel(f"fastaxis {d['fastaxis']} // pixelX")
        axmosaic.set_ylabel(f"slowaxis {d['slowaxis']} // pixelY")
    title = ''
    title += f'{folder}'
    title += 'roicenter X,Y = (%d,%d)'%(d['roicenter'][0],d['roicenter'][1])
    #axmosaic.set_title(title)
    figmosaic.suptitle(title)
    axmosaic.format_coord = format_coord

#   COLLECT ROI counters multiprocessing

(setting collector, boxsizes)

In [ ]:
def get_largest_index_in_folder(folder_path: str, filename_prefix: str = 'img_', filename_suffix: str = 'tif') -> int:
    """
    Find the filename with the largest index in a folder.

    Parameters
    ----------
    folder_path : str
        folder path
    filename_prefix : str, optional
        prefix of filename, default is 'img_'

    Returns
    -------
    largest_index : int
        largest index in filename
    """
    import re
    def extract_number(path: Path):
        return int(re.search(r'\d+$', path.stem).group())

    files = sorted(Path(folder_path).glob(f'{filename_prefix}*.{filename_suffix}'), key=extract_number)
    
    #files = sorted(Path(folder_path).glob(f'{filename_prefix}*.{filename_suffix}'))
    largest_index = int(Path(files[-1]).stem.split('_')[1])
    return largest_index

d['mapdimensions'],d['folder'],get_largest_index_in_folder(d['folder'], filename_prefix=d['prefix'],
                                                         filename_suffix=d['suffix'])

In [ ]:
d['folder'], max(d['listindices']), get_largest_index_in_folder(d['folder'], filename_prefix=d['prefix'], filename_suffix='h5')

In [ ]:
# checking if we have more images than expected
if get_largest_index_in_folder(d['folder'], filename_prefix=d['prefix'], filename_suffix='h5') != max(d['listindices']):
    GT.printyellow(f'\nBe careful, this folder created by a scan may contain unexpected additional images')
    GT.printyellow(f"\nPlease use d['listindices'] to properly read the right number of images")
else:
    GT.printgreen(f'\nThis folder containing images created by a scan contains {max(d["listindices"])+1} images as expected from dict `d`')

## Filtering rois position far from border

In [ ]:
import LaueTools.dict_LaueTools as DictLT
def filter_points_far_from_border(points, min_distance_x, min_distance_y, CCDLabel=None):
    """
    Filters 2D points to keep only those far from the borders, with separate tolerances for X and Y.

    Parameters:
    - points: 2D array of shape (N, 2) representing the points.
    - tuple of maximum dimensions
    - min_distance_x: Minimum distance from X borders to keep a point.
    - min_distance_y: Minimum distance from Y borders to keep a point.

    Returns:
    - filtered_points: Filtered 2D array of points.
    - kept_indices: Indices of the points that are kept.
    """
    framedim = DictLT.dict_CCD[CCDLabel][0]
    
    dimX, dimY = framedim 
    
    x_min, x_max = 0, dimX
    y_min, y_max = 0, dimY

    # Calculate the distance from each point to the X and Y borders
    distance_to_x_borders = np.minimum(
        np.abs(points[:, 0] - x_min),
        np.abs(points[:, 0] - x_max)
    )
    distance_to_y_borders = np.minimum(
        np.abs(points[:, 1] - y_min),
        np.abs(points[:, 1] - y_max)
    )

    # Determine which points are far enough from both X and Y borders
    far_from_x = distance_to_x_borders >= min_distance_x
    far_from_y = distance_to_y_borders >= min_distance_y
    kept_mask = far_from_x & far_from_y

    # Get the indices and filtered points
    kept_indices = np.where(kept_mask)[0]
    filtered_points = points[kept_mask]

    return filtered_points, kept_indices


In [ ]:
maxdistancefromborder = 100  # will be the half boxsize used in the next cell!
if 1:
    pk0 = d['peaklist']
    print('Checking and keeping peaks far from detector frame border')
    print('before',len(pk0))
    pk, totakeindices = filter_points_far_from_border(pk0, maxdistancefromborder, maxdistancefromborder, d['CCDLabel'])
    print('after', len(pk))
    print(f"updating peaks list of d['peaklist']")
    d['peaklist'] = pk
    if d['intensities'] is not None:
        d['intensities']= np.take(d['intensities'],totakeindices , axis=0)


In [ ]:
if __name__=='__main__':
    import itertools

    # True will deduce the last completed lines and modify map dimensions accordingly, or if scan has been aborted
    # False: consider the scan completed (all images stored) or one can define listindcies for; dict of scan
    ONLINE=False  # True  
    # -----------------INPUT ---------------------------------------
    # advised for 2D map
    #collector = 'roimax','roiXYmax' 'fitpeakXY' 'XYcenterofmass'
    # advised for daxm
    #collector = 'roimax', 'pixelval' 'sum'  'nbhotpixels'
    collector = 'nbhotpixels'

    threshold = 400 # for nbhotpixels  abov local background

    nbcpus= 64 # > 1 !!

    computerrorbars = False
    
    boxsize_X=21 # along pixel X    half size
    boxsize_Y=21  # along pixel Y   half size
    
    if  d['scantype'] == 'daxm':
        boxsize_X=1
        boxsize_Y=1
        
    elif collector == 'pixelval':
        boxsize_X=None  # meaningless
        boxsize_Y=None

    
    dictparam=d

    folder=dictparam.get('folder', None)
    
    # images to be handled
    listindices = None#  None means take indices in d[['listindices']]
    nbcompletelines = None #10  # None all lines when scan is completed
    
    if ONLINE:
        nbcompletelines = get_largest_index_in_folder(folder, filename_prefix=d['prefix'],
                                                         filename_suffix=d['suffix'])//d['mapdimensions'][0]
        print('nbcompletelines',nbcompletelines)

    
    #---------------------Multiprocessing-----------------------------------------------
    if collector == 'fitpeakXY':
        computerrorbars = False # True
        
    maxnbcpus = cpu_count()
    if nbcpus is None:
        nbcpus = maxnbcpus
    else:
        nbcpus = max(2,min(nbcpus,maxnbcpus))
    print('nbcpus', nbcpus)
    
    peaklist=dictparam.get('peaklist', None)
    prefix=dictparam.get('prefix', None)
    
    CCDLabel=dictparam.get('CCDLabel', 'sCMOS')
    
    nbpeaks = len(dictparam['peaklist'])
    
    if listindices is None:
        listindices=dictparam['listindices']
        if nbcompletelines is not None:
            listindices=dictparam['listindices'][:d['nbimagesperline']*nbcompletelines]
            
    if not isinstance(folder, list):  # iteration over images on the same folder
        nbimages = len(listindices)
        listfolders = itertools.repeat(folder)
        
    else: # iteration over a list of folder
        print('Multiple folders screened')
        nbimages = len(folder)
        listfolders = folder
        listindices = itertools.repeat(dictparam['listindices'])
    
    
    #----------------------------------
    
    t00 = time.time()
    print(f'using {collector} as collector: {nbimages} images, {nbpeaks} roi positions, {nbcpus} cpu(s)')
    
    args_pixelvalue = zip(listindices,
                   itertools.repeat(peaklist),
                   itertools.repeat(prefix),
                   listfolders,
                   itertools.repeat(CCDLabel),
                   )

    args_roimax = zip(listindices,
                   itertools.repeat(peaklist),
                   itertools.repeat(prefix),
                   listfolders,
                  itertools.repeat(boxsize_X),
                  itertools.repeat(boxsize_Y),
                   itertools.repeat(CCDLabel),
                   )

    args_roimax_errorbars = zip(listindices,
                   itertools.repeat(peaklist),
                   itertools.repeat(prefix),
                   listfolders,
                  itertools.repeat(boxsize_X),
                  itertools.repeat(boxsize_Y),
                   itertools.repeat(CCDLabel),
                    itertools.repeat(computerrorbars)
                   )

    args_nbhotpixels =  args_roimax = zip(listindices,
                   itertools.repeat(peaklist),
                   itertools.repeat(prefix),
                   listfolders,
                  itertools.repeat(boxsize_X),
                  itertools.repeat(boxsize_Y),
                   itertools.repeat(CCDLabel),
                    itertools.repeat(threshold)
                   )
    
    #print('args_pixelvalue',[elem for elem in args_pixelvalue])
    
    with multiprocessing.Pool(nbcpus) as pool:
    
        assert len(peaklist)>0

        if collector == 'pixelval': # 1 scalar /roi
            print("collectpixelvalue_singlefile")
            allresults = pool.starmap(collectpixelvalue_singlefile,
                                      tqdm(args_pixelvalue, total=nbimages,
                                      desc='pixel values progress bar:'))
            
        elif collector == 'sum': # 1 scalar /roi
            allresults = pool.starmap(collectroissum_singlefile,
                                      tqdm(args_roimax, total=nbimages,
                                      desc='sum values progress bar:'), chunksize=1)

        elif collector == 'nbhotpixels': # 1 scalar /roi
            allresults = pool.starmap(collectrois_nbhotpixels,
                                      tqdm(args_nbhotpixels, total=nbimages,
                                      desc='sum values progress bar:'), chunksize=1)

            
        elif collector == 'roimax':

            allresults = pool.starmap(collectroismax_singlefile,
                                      tqdm(args_roimax, total=nbimages,
                                      desc='max values progress bar:'), chunksize=1)
            #collectroismax_singlefile.__defaults__=(peaklist,prefix,folder,boxsize_X,boxsize_Y,CCDLabel)
            #allresults = pool.map(collectroismax_singlefile, tqdm(listindices))
            
        elif collector == 'roiXYmax': #  2 values/roi

            allresults = pool.starmap(collectroisXYmax_singlefile,
                                      tqdm(args_roimax, total=nbimages,
                                      desc='pixel positions progress bar:'))

        elif collector == 'fitpeakXY':  # 7 values/roi
            allresults = pool.starmap(collectroisfitpeak_singlefile,
                                      tqdm(args_roimax_errorbars, total=nbimages,
                                      desc='fitted positions progress bar:'), chunksize=1)
        elif collector == 'XYcenterofmass': #  2 values/roi
            allresults = pool.starmap(collectroisXYcenterofmass_singlefile,
                                      tqdm(args_roimax, total=nbimages,
                                      desc='pixel positions progress bar:'))

    allresults = np.array(allresults)

    elapsedtime=time.time()-t00
    print(f'total time is {elapsedtime:.3f} sec for {nbimages} images, {nbpeaks} roi positions and {nbcpus} cpu(s)')

    children = active_children()
    print(f'Active children: {len(children)}')

    d['allresults']=allresults
    GT.printgreen('Results are in allresults or allresults key...\n\n!!! ----- Collection done ;-) ----!!')


In [ ]:
allresults.shape

## pickling results 

In [ ]:
#'/data/visitor/blc16859/bm32/20260210/PROCESSED_DATA/Texier/Texier_T2/scan0007'

In [ ]:
!ls /data/visitor/blc16859/bm32/20260210/PROCESSED_DATA/Texier/Texier_T2/scan0007/*pickle

In [ ]:

# Define the folder path starting from the home directory
#save_path = Path("~/LaueTutorialsResults").expanduser()

save_path = Path(d['imagefolder'].replace('RAW_DATA', 'PROCESSED_DATA'))
save_path.mkdir(parents=True, exist_ok=True)

#---------------------------------------------
# use genfolder to write output  results
import pickle

nbcounters = allresults.shape[1]

if 1: # SAVE
    dictresults=copy.copy(d)
    dictresults['allresults']=allresults
    with open(os.path.join(save_path,'%s_%dcounters.pickle'%(collector,nbcounters)), 'wb') as f:
        pickle.dump(dictresults, f)

if 0: #LOAD
    if input('Are you sure to load a previous file and overwrite dictresults?') in ('y','yes','Y','YES','o','O'):
        
        folder = '/data/visitor/ihma346/bm32/20230304/SiCnuit/SiCnuit_bulle2/scan0001'
        collector = 'fitpeakXY'
        #folder = d['folder']

        picklefile = save_path
        picklefile = '/data/visitor/blc16859/bm32/20260210/PROCESSED_DATA/Texier/Texier_T2/scan0007/roimax_4465counters.pickle'

        #with open(os.path.join(folder,'%s_20counters.pickle'%collector), 'rb') as f:
        with open(os.path.join(picklefile,'fitpeakXY_20counters'+'.pickle'), 'rb') as f:
            dictresults=pickle.load(f)
        allresults = dictresults['allresults']
        d=dictresults

## arrange data from collected data

- just execute the cell
- [OPTION] SET scalar X or Y component for `collector` = 'roiXYmax' or 'fitpeakXY'
- build `tr` array

In [ ]:
#peak component selection  (only if 'roiXYmax' or 'fitpeakXY' collected data)
peakposcomponent='X'
#peakposcomponent='Y' 

print('(nb images, nb rois, nb vals / roi)',allresults.shape)
print('collector',collector)

if collector in ('sum','roimax','pixelval','nbhotpixels'):
    tr = np.transpose(allresults)
    
    collectortitle=collector #just for visualisation
    
    spatialcoord = ''
    
elif collector in ('fitpeakXY',):
    # bke, amp, xfit, yfit, std1, std2, orientangle,  [errorbars] ?
    bkg  = allresults[:,:,0]
    amp = allresults[:,:,1]
    xfit, yfit = np.transpose(allresults[:,:,2:4])
    
    spatialcoord = peakposcomponent
    
    if peakposcomponent == 'X':
        tr = xfit
    if peakposcomponent == 'Y':
        tr = yfit
    collectortitle=collector #just for visualisation
else:
    tr_x = np.transpose(allresults[:,:,0])
    tr_y = np.transpose(allresults[:,:,1])
    
    spatialcoord = peakposcomponent
    
    # selecting X value
    if peakposcomponent=='X':
        tr = tr_x
    else: # selecting Y value
        tr = tr_y
    collectortitle=collector+'_%s'%peakposcomponent #just for visualisation

print('nb rois',tr.shape[0])
print('nb images',tr.shape[1])
print('map dimensions:',d['mapdimensions'])
print('map nb of dimensions:',len(d['mapdimensions']))
print('Chosen scalar to map in 2D:', collectortitle)
print('scan type', d['scantype'])
if d['scantype']=='map':
    GT.printgreen('Your data are 2D, so use VISUALIZE 2D case')
elif d['scantype']=='daxm':
    GT.printgreen('Your data are 1D, so use VISUALIZE 1D case')


# VISUALIZE: 2D case (amesh dmesh 2D map)

this section is useful only if d['scantype']=='map'

In [ ]:
# Advice
if not d['scantype']=='map':
    print(f'2D visualisation is not appropriate for your scan since d["scantype"]="{d["scantype"]}" is not "map".\nGo better to the next section!')

In [ ]:
#GT.getmotorspositionfromimageindex(200,d)

### PLOT multiple maps from several roi counters

In [ ]:
# for 2D dimension data (sample map)

# select counters
list_idx_counters=range(0,60)
list_idx_counters=range(0,5)
#list_idx_counters = sim_list_idx_counters

autocontrast = False

#------------------end of user inputs------------------------------------
print('fastaxis', d['fastaxis'])
nbcounters = len(list_idx_counters)

print('nbcounters',nbcounters)
print('highest selected index counter',max(list_idx_counters))

if max(list_idx_counters)>len(tr):
    msg='maximum counter index is %d'%(len(tr)-1)
    raise ValueError(msg)
# select displayed layout of plots
nbcols = 6
#nbcols = 1

nbimagesperline = d['nbimagesperline']
numrows, numcols = d['mapdimensions']

print('nbimagesperline',nbimagesperline)
print(" d['mapdimensions']", d['mapdimensions'])
#----------------------------------
nrows=nbcounters//nbcols
if nbcounters%nbcols != 0:
    nrows +=1

print('nrows',nrows)

_list_idx_counters=list(list_idx_counters)
if nbcounters<nrows*nbcols:
    _list_idx_counters = _list_idx_counters +['None'] * (nrows*nbcols - nbcounters)

print(nbcounters,nrows, nbcols)
fig2, axs = plt.subplots(ncols=nbcols, nrows=nrows, sharex=True, sharey=True, figsize=(8,16))
#fig2.suptitle('%s'%collectortitle,fontsize=16)

missingplot=np.zeros_like(tr[0]).reshape((-1,nbimagesperline))
# ss = missingplot.shape

def format_coord(x, y):
    fastdim, slowdim = d['mapdimensions']
    col = int(x + 0.5)
    row = int(y + 0.5)
    if col >= 0 and col < fastdim and row >= 0 and row < slowdim:
        #cnt_idx = fig.gca()
        img_idx = 0+ nbimagesperline*row+col
        return "x=%1.4f, y=%1.4f, imageid=%d, intensity" % (x, y,img_idx)
    else:
        return "x=%1.4f, y=%1.4f, intensity" % (x, y)

k=0
axsflat=axs.flat
for ax, _idx in zip(axsflat,_list_idx_counters):
    if k<nbcounters:
        roidata = tr[_idx]
        vmin,vmax=None,None
        if autocontrast:
            #print('mean',np.mean(roidata))
            vmin, vmax = 1010, max(1010, 0.75*np.amax(roidata))
            
        if not 'fastaxis' in d:
            transdata = roidata.reshape((-1,nbimagesperline))
            xlabel = 'xech'
            ylabel = 'yech'
        else:
            if d['fastaxis'] in ('xech', 'xps'):
                transdata = roidata.reshape((-1,nbimagesperline)) # .T ???
                xlabel = 'xech'
                ylabel = 'yech'
            elif d['fastaxis'] in ('yech','yps'):
                transdata = roidata.reshape((-1,nbimagesperline))
                xlabel = 'yech'
                ylabel = 'xech'
                
            
        ax.imshow(transdata, origin='lower', vmin=vmin,vmax=vmax)
        ax.format_coord = format_coord
        #ax.set_title('%d'%_idx, fontsize=6, color='red')
        ax.text(0.5, 0.05, '%d'%_idx,
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=7, color='red',
                        transform=ax.transAxes)
        #ax.set_xlim(20,30)
    else:
        ax.imshow(missingplot, origin='lower')
        ax.text(0.5, 0.5, 'no data',
                        horizontalalignment='center',
                        verticalalignment='center',
                        fontsize=5, color='red',
                        transform=ax.transAxes)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    k+=1

plt.tight_layout(pad=0)
#plt.subplots_adjust(left=0.05, bottom=0.05, right=0.95, top=0.95, wspace=0.1, hspace=0.1)
plt.show()

## save figure and peaks/rois coordinates

In [ ]:
outputfolder = save_path
#outputfolder =  d['folder']
 
# ----  output filename business  -------
nameout = d.get('title','')
tt =time.asctime().strip()
tt = tt.replace(' ','_')
tt=tt.replace(':','_')
coltime="%s_%s_%s"%(collector,spatialcoord,tt)

if nameout != '':
    nameout+='_'+coltime
else:
    nameout=coltime
#---------------------------
#print(nameout)
#--------------------------
lcnt = tuple(list_idx_counters)
xsel, ysel = np.take(d['peaklist'],lcnt , axis=0).T
ct_indices = np.array(lcnt)
selected_peaks=np.array([ct_indices,xsel,ysel]).T
coordfile = "roicoordinates_%s.dat"%nameout

with open(os.path.join(d['folder'],coordfile),'w') as f:
    np.savetxt(f, selected_peaks, header='roi_index, X, Y')
# -------------------

plt.savefig(os.path.join(outputfolder,nameout+'.png'))
GT.printgreen(f'output folder ---> {outputfolder}')
GT.printgreen(f'figure filename ---> {nameout}.png')
GT.printgreen(f'roi coordinates filename ----> {coordfile}')

###  PLOT of a single map from 1 counter

In [ ]:
# select counters
idx_counter = 5

outputfolder = save_path
# ----  output filename business  -------
nameout = d.get('title','')
tt =time.asctime().strip()
tt = tt.replace(' ','_')
tt=tt.replace(':','_')
coltime="%s_%s_%s"%(collector,spatialcoord,tt)

if nameout != '':
    nameout+='_'+coltime
else:
    nameout=coltime
#---------------------------

print('nbimagesperline',nbimagesperline)

xy = np.take(d['peaklist'],idx_counter, axis=0)

def format_coord2(x, y):
    
    if d['fastaxis']=='yech':
        col = int(y + 0.5)
        row = int(x + 0.5)
        nfast, nslow = d['mapdimensions']
    else:
        col = int(x + 0.5)
        row = int(y + 0.5)
        nslow, nfast = d['mapdimensions']
    
    if col >= 0 and col < nfast and row >= 0 and row < nslow:
        #cnt_idx = fig.gca()
        cnt_idx = 1
        img_idx = 0+ nbimagesperline*row+col
        return "x=%1.4f, y=%1.4f, imageid=%d" % (x, y,img_idx)
    else:
        return "x=%1.4f, y=%1.4f" % (x, y)

fig3, ax = plt.subplots()
dd = tr[idx_counter]
mini = np.amin(dd)
avg = np.mean(dd)
vmin, vmax = avg-0.5,avg+0.5
vmin, vmax = None, None  # default

if autocontrast:
    #print('mean',np.mean(roidata))
    #vmin, vmax = 1010, max(1010, 0.75*np.amax(tr[idx_counter]))
    vmin, vmax = np.amin(dd), np.amax(dd)
    
if not 'fastaxis' in d:
    transdata = dd.reshape((-1,nbimagesperline))
    xlabel = 'xech'
    ylabel = 'yech'
else:
    if d['fastaxis']=='xech':
        transdata = dd.reshape((-1,nbimagesperline))
        xlabel = 'xech'
        ylabel = 'yech'
    elif d['fastaxis']=='yech':
        transdata = dd.reshape((-1,nbimagesperline)).T
        ylabel = 'yech'
        xlabel = 'xech'
    
plt.imshow(transdata, origin='lower', vmin=vmin, vmax=vmax)
plt.title("%s %s\nroi #%d centered on X,Y =%.1f, %.1f"%(collectortitle,spatialcoord,idx_counter,xy[0],xy[1]))
ax.format_coord = format_coord2
ax.set_xlabel(xlabel)
ax.set_ylabel(ylabel)

plt.colorbar()

roicounterfile = "roi_%d_x_%d_y_%d_%s"%(idx_counter,int(xy[0]),int(xy[1]),nameout)

# save plot in image
plt.savefig(os.path.join(outputfolder,roicounterfile+'.png'))
# save data in file
header = '%s %s %s mapdimensions=%s'%(roicounterfile, collectortitle,d['folder'],str(d['mapdimensions']))
with open(os.path.join(outputfolder,roicounterfile+'.dat'),'w') as f:
    np.savetxt(f, dd.reshape((-1,nbimagesperline)),
               header=header)

print('saving folder ---->', outputfolder)
print('plot saved in ---->',roicounterfile+'.png')
print('data saved in ---->',roicounterfile+'.dat')

## GUI for 2D case, to browse over maps from all roi counters

In [ ]:
print('nb of Roi counters', ', nb of images')
print(tr.shape)
print('scan full command',d.get('fullcommand',None))
print('nbimagesperline',d['mapdimensions'][0])
print("d['mapdimensions']",d['mapdimensions'])
print("d['fastaxis']",d['fastaxis'])

In [ ]:
#------------GUI  to browse counters-------------------
# ----  well tested with fast axis is yech: amesh yech ... ...  xech ... ... 

# ------------INPUT---------------
idx_counter0 = 0
autocontrast0 = True
#---------------------------------

nbmaxcounters = tr.shape[0]
#print('nbmaxcounters',nbmaxcounters)

# ----  output filename business  -------
nameout = d.get('title','')
tt =time.asctime().strip()
tt = tt.replace(' ','_')
tt=tt.replace(':','_')
coltime="%s_%s_%s"%(collector,spatialcoord,tt)

if nameout != '':
    nameout+='_'+coltime
else:
    nameout=coltime
#---------------------------
nbimagesperline = d['nbimagesperline']


currentroi_idx = idx_counter0

dd= np.copy(tr[idx_counter0])
currentdd = dd
xy = np.take(d['peaklist'],idx_counter0, axis=0)

def format_coord3(x, y):  
    if d['fastaxis']=='yech':
        col = int(x + 0.5)
        row = int(y + 0.5)
        nfast, nslow = d['mapdimensions']
    else:
        col = int(x + 0.5)
        row = int(y + 0.5)
        nslow, nfast = d['mapdimensions']
    if col >= 0 and col < nfast and row >= 0 and row < nslow:
        #cnt_idx = fig.gca()
        cnt_idx = 1
        img_idx = 0+ nbimagesperline*row+col
        return "x=%1.4f, y=%1.4f, imageid=%d" % (x, y,img_idx)
    else:
        return "x=%1.4f, y=%1.4f" % (x, y)

fig55, ax55 = plt.subplots(figsize=(5,5))
mini = np.amin(dd)
avg = np.mean(dd)
vmin, vmax = avg-0.5,avg+0.5
vmin, vmax = None, None  # default

if not 'fastaxis' in d:
    tdata = dd.reshape((-1,nbimagesperline))
    xlabel = 'xech'
    ylabel = 'yech'
else:
    if d['fastaxis'] in ('xech','xps'):
        tdata = dd.reshape((-1,nbimagesperline))
        xlabel = 'xech'
        ylabel = 'yech'
    elif d['fastaxis'] in ('yech','yps'):
        tdata = dd.reshape((-1,nbimagesperline))
        ylabel = 'xech'
        xlabel = 'yech'
 
ax55.imshow(dd.reshape((-1,nbimagesperline)), origin='lower', vmin=vmin, vmax=vmax)
ax55.format_coord = format_coord3

roicounterfile = "roi_%d_x_%d_y_%d_%s"%(idx_counter0,int(xy[0]),int(xy[1]),nameout)

from IPython.display import display

def plotmap2D(idx_counter=idx_counter0, vmin = 1000, vmax=1200, autocontrast=autocontrast0,
               scale='linear',printstatistics=False):
    xlim,ylim = ax55.get_xlim(), ax55.get_ylim()
    ax55.clear()
    ax55.set_xlim(xlim)
    ax55.set_ylim(ylim)

    currentroi_idx = idx_counter

    dd= np.copy(tr[idx_counter])

    currentdd = dd

    posmax = np.argmax(dd)
    if printstatistics:
        print('idx_counter', idx_counter,
              'max intensity', np.amax(dd), 
              'at imageindex, (col, row)',posmax ,posmax//nbimagesperline,posmax%nbimagesperline)
#     _vmin=np.amin(dd)
#     vmax=max(_vmin,vmax)
    if autocontrast:
        _max = np.amax(dd)
        _min = np.amin(dd)
        #print('mean',np.mean(roidata))
        vmin, vmax = _min, _max
        
    if not 'fastaxis' in d:
        tdata = dd.reshape((-1,nbimagesperline))
        xlabel = 'xech'
        ylabel = 'yech'
    else:
        if d['fastaxis']in ('xech','xps'):
            tdata = dd.reshape((-1,nbimagesperline))
            xlabel = 'xech'
            ylabel = 'yech'
        elif d['fastaxis'] in ('yech','yps'):
            tdata = dd.reshape((-1,nbimagesperline))
            xlabel = 'yech'
            ylabel = 'xech'
 
    xy = np.take(d['peaklist'],idx_counter, axis=0)
    
    if scale=='log':
        _vmax = np.log10(vmax)
        _vmin = np.log10(vmin)
        _tdata = np.log10(tdata)
    elif scale=='sqrt':
        _vmax = np.sqrt(np.fabs(vmax-1000))
        _vmin = np.sqrt(np.fabs(vmin-1000))
        _tdata = np.sqrt(np.fabs(tdata-1000))
    elif scale=='atan':
        nor = 60000/np.pi*2
        _vmax = np.arctan(np.fabs(vmax-1000)/nor)
        _vmin = np.arctan(np.fabs(vmin-1000)/nor)
        _tdata = np.arctan(np.fabs(tdata-1000)/nor)
    else:
        _vmax = vmax
        _vmin = vmin
        _tdata = tdata
    
    im = ax55.imshow(_tdata, origin='lower', vmax=_vmax, vmin=_vmin, aspect=1)
    txttitle = f"{collectortitle} {spatialcoord}\nroi #{idx_counter} centered on X,Y ={xy[0]:.1f}, {xy[1]:.1f}"
    ax55.set_title(txttitle)
    ax55.set_xlabel(xlabel)
    ax55.set_ylabel(ylabel)
    plt.show()

if collector == 'sum':
    vmax0 = 30000000
    vminmax0 = vmax0/2.
else:
    vmax0 = 2000
    vminmax0 = 0
    
ppi = interactive(plotmap2D, idx_counter=(0,nbmaxcounters-1),
            vmin=(0,vminmax0), vmax=(2,vmax0), autocontrast=[False, True],
                 scale=['linear','log','sqrt','atan'],
                 printstatistics=[False, True])

buttonsave = widgets.Button(description="Save Plot")
output = widgets.Output()

def on_button_clicked(b):
    """save current plot in .png format and data in npy"""
    fname = os.path.join(save_path,'map_roi_%d'%currentroi_idx)
    plt.savefig(fname)
    np.save(fname, currentdd)
    with output:
        output.clear_output()
        print(f'plot saved in :\n{fname}')
        
buttonsave.on_click(on_button_clicked)
if 0:
    from ipywidgets import TwoByTwoLayout
    TwoByTwoLayout(top_left=ppi, top_right=buttonsave,
                   bottom_right=output,
                   justify_items='right',
                   width="95%",
                   align_items='center')
if 1:
    from ipywidgets import HBox, VBox, Box

    # Create a 2x2 layout with ppi taking more space
    layout = VBox([
        HBox([
            Box([ppi], layout={'width': '70%', 'justify_content': 'center'}),  # Top-left (ppi takes 70%)
            Box([buttonsave], layout={'width': '30%', 'justify_content': 'flex-end'})  # Top-right (30%)
        ]),
        HBox([
            Box([], layout={'width': '70%'}),  # Bottom-left (empty, 70%)
            Box([output], layout={'width': '30%', 'justify_content': 'flex-end'})  # Bottom-right (30%)
        ])
        ])

    # Display the layout
    display(layout)

In [ ]:
d['folder']

In [ ]:
d['fullcommand']

In [ ]:
fdmapl(xech,-0.075,.075,300,yech,-0.05,0.05,500,.25,0, mon, cam)